# El Parser Avanzado
Diego Torres Durante

Carga en un notebook nuevo su archivo limpio de la Play Store y resuelvan los siguientes tres retos de ingeniería. Documenten su código y respondan a las preguntas de negocio en celdas Markdown.

In [16]:
import pandas as pd
import numpy as np

url = "../datasets/playstore_limpio.csv"
df = pd.read_csv(url)

print("Datos listos. Total de apps:", len(df))

Datos listos. Total de apps: 9636


## Reto 1: La Trampa de las Unidades de Medida

El equipo de infraestructura quiere saber si las aplicaciones más pesadas tienen menos descargas. Para averiguarlo, necesitan que la columna `Size` sea completamente numérica. Sin embargo, si revisan la columna, encontrarán valores como `19M` (Megabytes), `201k` (Kilobytes) y un texto molesto que dice `Varies with device`.

### Tu misión algorítmica

Escribe una función pura en Python que reciba un string. Si el string termina en `M`, quítale la `M` y conviértelo a flotante, dejándolo como Megabytes. Si termina en `k`, quítale la `k`, conviértelo a flotante y divídelo entre 1024 para pasarlo también a Megabytes. Si dice `Varies with device`, devuélvelo como un valor nulo de Numpy (`np.nan`).

Aplica esta función a toda la columna utilizando el método `.apply()`.


In [17]:
def convertir_size(size):
    if size == "Varies with device":
        return np.nan
    elif size.endswith("M"):
        return float(size[:-1])
    elif size.endswith("k"):
        return float(size[:-1]) / 1024

df["Size_MB"] = df["Size"].apply(convertir_size)
print("Peso promedio en MB:", df["Size_MB"].mean())

Peso promedio en MB: 20.41973193945847


### Pregunta

Una vez convertida la columna a valores numéricos (Megabytes), ejecuta el método `.mean()`. ¿Cuál es el peso promedio en Megabytes de las apps en la Play Store?

### Respuesta
20.42MB.



----

## Reto 2: El Tipo de Dato Cronológico

Ningún análisis de software está completo sin analizar el tiempo. La columna `Last Updated` tiene fechas escritas como texto, por ejemplo `January 7, 2018`. Para un modelo matemático o una serie de tiempo, eso es texto inservible.

### Tu misión algorítmica

Investiga y utiliza la función `pd.to_datetime()` de Pandas para sobrescribir la columna `Last Updated`, convirtiéndola del tipo string (`object`) al tipo nativo `datetime64`.

Ahora que es un objeto de tiempo, Pandas te permite extraer componentes específicos. Crea una nueva columna llamada `Year_Updated` extrayendo únicamente el año (`df['Last Updated'].dt.year`).


In [18]:
df["Last Updated"] = pd.to_datetime(df["Last Updated"])
df["Year_Updated"] = df["Last Updated"].dt.year
print(df["Year_Updated"].value_counts())

Year_Updated
2018    6271
2017    1785
2016     779
2015     448
2014     203
2013     108
2012      26
2011      15
2010       1
Name: count, dtype: int64


### Pregunta

Utilizando la sumarización categórica (`value_counts()`) sobre tu nueva columna `Year_Updated`, ¿en qué año se actualizó la mayor cantidad de aplicaciones en nuestro dataset?

### Respuesta
2018 (6271).



----

## Reto 3: La Decisión Arquitectónica

Al resolver el Reto 1, introdujiste intencionalmente valores `NaN` en las aplicaciones cuyo tamaño decía `Varies with device`.

### Tu misión algorítmica

Evalúa cuántos registros quedaron vacíos. Como ingenieros, decidan y apliquen la mejor técnica: ¿es matemáticamente más sano borrar esas filas con un `.dropna()` porque el peso de la app es crítico, o prefieren rellenar ese hueco imputando la mediana global del peso de las apps?


In [19]:
print("Registros vacíos:", df["Size_MB"].isna().sum())
median_size = df["Size_MB"].median()
df["Size_MB"] = df["Size_MB"].fillna(median_size)
print("Registros vacíos después de imputar:", df["Size_MB"].isna().sum())
print("Peso promedio en MB", df["Size_MB"].mean())

Registros vacíos: 1227
Registros vacíos después de imputar: 0
Peso promedio en MB 19.347605425374248


### Pregunta

Redacten una breve justificación técnica de 3 líneas explicando qué método eligieron y por qué lo consideran superior para no dañar al modelo de predicción.

### Respuesta
1227 apps con registros vacios de un total apps. si se utiliza la mediana, el promedio no varía mucho (+-1 MB), y puede que podamos utilizar más registros para entrenamiento, pero tal vez y depende del caso si conviene no rellenar. Lo que si es que la mediana es suceptible a outliers al imputar.



----